In [ ]:
%%javascript

// h/t ChatGPT

(async function() {
     console.log("Creating AI asisstant...\n\n");
 
    // Assuming ai.assistant.create() is defined 
    // and accessible in your environment
    const session = await ai.assistant.create();

    console.log("Session created...\n\n");

    console.log("Sending prompt...\n\n");

    // Prompt the model and wait for the result
    const result = await session.prompt("Write me a poem.");


    // Log the result to the console
    console.log(result);
})();

sdsd  $\circlearrowleft$  sdsd 

```mermaid

graph LR;
    A-->B;
    A-->C;
    B-->D;
    C-->D;
    D-->E;
```

In [1]:
%load_ext jupyter_anywidget_pglite
from jupyter_anywidget_pglite import pglite_headless


In [2]:
pg = pglite_headless()
pg.ready()

postgresWidget(headless=True, response={'status': 'initialising'})

In [3]:
%setwidget pg

In [4]:
%%pglite_magic -M -r
CREATE TABLE IF NOT EXISTS test  (
        id serial primary key,
        title varchar not null
      );


INSERT INTO test (title) VALUES ('dummy');

SELECT * FROM test;

{'status': 'completed',
 'response': [{'rows': [], 'fields': []},
  {'rows': [], 'fields': []},
  {'rows': [{'id': 1, 'title': 'dummy'}],
   'fields': [{'name': 'id', 'dataTypeID': 23},
    {'name': 'title', 'dataTypeID': 1043}],
   'affectedRows': 1}],
 'response_type': 'multi'}

In [6]:
pg.blocking_reply(timeout=5)

{'status': 'completed',
 'response': [{'rows': [], 'fields': []},
  {'rows': [], 'fields': []},
  {'rows': [{'id': 1, 'title': 'dummy'}],
   'fields': [{'name': 'id', 'dataTypeID': 23},
    {'name': 'title', 'dataTypeID': 1043}],
   'affectedRows': 1}],
 'response_type': 'multi'}

In [7]:
%pglite_query -r -q 'SELECT * FROM test LIMIT 1;'


{'status': 'completed',
 'response': {'rows': [{'id': 1, 'title': 'dummy'}],
  'fields': [{'name': 'id', 'dataTypeID': 23},
   {'name': 'title', 'dataTypeID': 1043}],
  'affectedRows': 0},
 'response_type': 'single'}

In [ ]:
pg.response

In [ ]:
bool(0 or False)

In [ ]:
pg.audio=True

In [ ]:
# Create a data dump
pg.create_data_dump(True)

In [ ]:
datadump = pg.file_package.copy()
datadump["file_info"]

In [ ]:
# Load a datadump into a new pglite widget
pg1 = pglite_panel(data=datadump)

In [ ]:
# Save datadump to file
pg1.save_datadump_to_file()

In [ ]:
# Seed a new widget with data dumped to a file
pg2 = pglite_panel(data="pgdata.tar.gz")

In [ ]:
%%pglite_magic -w pg2 -r
SELECT * FROM test;

In [ ]:
file_package["file_info"]

In [ ]:
pg.datadump

In [ ]:
datadump['file_content'] == file_package["file_content"]

In [ ]:
%%pglite_magic -w pg1
SELECT * FROM test;

In [ ]:
pg1.response

In [ ]:
pg.file_package["file_info"], pg.file_package["file_content"][:10]

In [ ]:
from jupyter_anywidget_pglite import load_datadump_from_file

dd = load_datadump_from_file("pgdata.tar.gz")
dd["file_info"]

In [ ]:
#pg.file_package["file_info"]

In [ ]:
pg.response

In [ ]:
pg.multiline

In [ ]:
pg.multiexec

In [ ]:
%load_ext jupyter_anywidget_graphviz
from jupyter_anywidget_graphviz import graphviz_panel

gv = graphviz_panel(title="dsd")

In [ ]:
%load_ext jupyter_anywidget_graphviz

In [ ]:
%setwidget gv

In [ ]:
%%graphviz_magic
  strict graph {
    a -- b
    a -- b
    b -- a [color=blue]
  }

In [ ]:
gv.svg

In [ ]:
#%pip install gtts moviepy

import re
import markdown
from bs4 import BeautifulSoup
from gtts import gTTS
from moviepy.editor import *

def parse_markdown(file_path):
    with open(file_path, 'r') as file:
        md_content = file.read()
    html_content = markdown.markdown(md_content)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    sections = []
    current_section = {'text': '', 'image': ''}
    
    for elem in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'img']):
        if elem.name.startswith('h'):
            if current_section['text'] or current_section['image']:
                sections.append(current_section)
                current_section = {'text': '', 'image': ''}
        elif elem.name == 'p':
            current_section['text'] += elem.get_text() + ' '
        elif elem.name == 'img':
            current_section['image'] = elem['src']
    
    if current_section['text'] or current_section['image']:
        sections.append(current_section)
    
    return sections

def create_audio(text, output_file):
    tts = gTTS(text)
    tts.save(output_file)

def create_video_scene(image_path, audio_path, output_path, duration):
    audio = AudioFileClip(audio_path)
    image = ImageClip(image_path).set_duration(audio.duration)
    video = image.set_duration(audio.duration).set_audio(audio)
    video.write_videofile(output_path, fps=24)

from IPython.display import Image, Audio
def mainVideo(markdown_file, outputvideo):
    sections = parse_markdown(markdown_file)
    clips = []

    for i, section in enumerate(sections):
        if section['text'] and section['image']:
            display(Image(section['image']))
            audio_file = f"audio_{i}.mp3"
            create_audio(section['text'], audio_file)
            audio = AudioFileClip(audio_file)
            display(Audio(audio_file))
            duration = audio.duration
            
            scene_file = f"scene_{i}.mp4"
            create_video_scene(section['image'], audio_file, scene_file, duration)
            
            clip = VideoFileClip(scene_file)
            clips.append(clip)

    final_video = concatenate_videoclips(clips)
    final_video.write_videofile(output_video, fps=24)

    # Clean up temporary files
    for clip in clips:
        clip.close()
    for i in range(len(sections)):
        os.remove(f"audio_{i}.mp3")
        os.remove(f"scene_{i}.mp4")

In [ ]:
mainVideo("../../../../Downloads/script.md", "dummymovie.mp4")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipykernel.comm import Comm
import threading
import uuid

class MessageProcessingWidget(widgets.DOMWidget):
    _view_name = traitlets.Unicode('MessageProcessingView').tag(sync=True)
    _view_module = traitlets.Unicode('custom').tag(sync=True)
    _view_module_version = traitlets.Unicode('0.1.0').tag(sync=True)
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.comm = Comm(target_name='message_processing_widget')
        self.comm.on_msg(self._handle_message)
        self.responses = {}
        self.events = {}

    def _handle_message(self, msg):
        data = msg['content']['data']
        if 'response' in data:
            msg_id = data['msg_id']
            self.responses[msg_id] = data['response']
            if msg_id in self.events:
                self.events[msg_id].set()

    def send_message(self, message, timeout=10):
        msg_id = str(uuid.uuid4())
        event = threading.Event()
        self.events[msg_id] = event
        
        self.comm.send({'msg_id': msg_id, 'message': message})
        
        if event.wait(timeout):
            response = self.responses.pop(msg_id, None)
            del self.events[msg_id]
            return response
        else:
            del self.events[msg_id]
            raise TimeoutError("Timeout waiting for response")

# JavaScript code for the widget
js_code = """
require.undef('custom');
define('custom', ["@jupyter-widgets/base"], function(widgets) {
    var MessageProcessingView = widgets.DOMWidgetView.extend({
        render: function() {
            this.el.textContent = 'Message Processing Widget';
            this.comm = this.model.widget_manager.kernel.comm_manager.new_comm('message_processing_widget');
            this.comm.on_msg(this.handle_kernel_message.bind(this));
        },
        handle_kernel_message: function(msg) {
            var data = msg.content.data;
            console.log('Received message:', data.message);
            
            // Simulate processing time (replace with actual processing logic)
            setTimeout(() => {
                var response = `Processed: ${data.message}`;
                this.comm.send({'msg_id': data.msg_id, 'response': response});
            }, 2000);  // 2 second delay for demonstration
        }
    });
    
    return {
        MessageProcessingView: MessageProcessingView
    };
});
"""
from IPython.display import Javascript
# Display the JavaScript code
display(Javascript(js_code))

# Create and display the widget
w = MessageProcessingWidget()
display(w)

# Function to send a message and wait for the response
def send_and_wait(message, timeout=10):
    print(f"Sending message: {message}")
    try:
        response = w.send_message(message, timeout)
        print(f"Received response: {response}")
    except TimeoutError:
        print("Timeout waiting for response")
    except Exception as e:
        print(f"Error while waiting for response: {e}")


In [ ]:
response = w.send_message("hello, timeout")

In [ ]:

# Example usage
send_and_wait("Hello, Widget!")

In [ ]:

# Wait for the response
response = await w.wait_for(msg_id)
print(f"Received response: {response}")

In [ ]:
import asyncio

async def func():
    print("started.")
    await asyncio.sleep(2)
    print("done.")

await func()

In [ ]:
import asyncio
from anywidget import AnyWidget
from traitlets import Unicode, observe

class AsyncioMessageWidget(AnyWidget):
    message = Unicode('').tag(sync=True)
    reply = Unicode('').tag(sync=True)
    last_query = ''
    last_response = ''
    def __init__(self):
        super().__init__()
        self._reply_future = None
        self.observe(self._on_reply_change, names=['reply'])

    @observe("message")
    def _observe_message(self, change):
        print(f"Old message value: {change['old']}")
        print(f"New message value: {change['new']}")
        print(f"------")

    @observe("reply")
    def _observe_reply(self, change):
        print(f"Old reply value: {change['old']}")
        print(f"New reply value: {change['new']}")
        print(f"------")
        if not change['new']:
            self.last_response=change["old"]
        
    def sendValue(self, msg):
        print("Sending message:", msg)  # Debugging
        self._reply_future = asyncio.Future()  # Initialize the future
        self.message = ''
        self.message = msg
        self.reply=""
        
    async def sendGetValue(self, msg, timeout=5):
        print("Sending message:", msg)  # Debugging
        self.message = ''
        self.reply=''
        #self.reply="weww"
        self._reply_future = asyncio.Future()  # Initialize the future
        #try:
        self.message = msg
        print("Waiting for reply...")  # Debugging
        reply = await asyncio.wait_for(self._reply_future, timeout)
        print("Received reply:", reply)  # Debugging
        return reply
        #except asyncio.TimeoutError:
        #    print("Timeout waiting for reply")  # Debugging
        #    raise TimeoutError("Timeout waiting for reply")
        
        
    async def get_reply_async(self, timeout=None):
        try:
            print("Waiting for reply...")  # Debugging
            reply = await asyncio.wait_for(self._reply_future, timeout)
            print("Received reply:", reply)  # Debugging
            return reply
        except asyncio.TimeoutError:
            print("Timeout waiting for reply")  # Debugging
            raise TimeoutError("Timeout waiting for reply")

    def _on_reply_change(self, change):
        display(f"reply change event {change}")
        if self._reply_future and not self._reply_future.done():
            print("Reply changed:", change['new'])  # Debugging
            self._reply_future.set_result(change['new']) 
            self.done = change["new"]
            # Set the result of the future
        else:
            print("No active future or future already done")  # Debugging
    _esm = """
    export function render({ model, el }) {
        const processMessage = (msg) => {
        console.log("p1")
            // Simulate some processing
            return new Promise(resolve => {
                setTimeout(() => {
                    resolve(`Processed: ${msg} ${Math.random()}`);
                    console.log("p2")
                }, 70);
            });
        };

        model.on('change:message', async () => {
        console.log("c1")
            const msg = model.get('message');
            console.log(`c2 ${msg}`)
            if (msg){
            console.log(`c3 ${msg}`)
            const reply = await processMessage(msg);
            console.log(`c4 ${msg} ${reply}`)
            model.set('reply', reply);
            model.save_changes();
            }
        });

        el.textContent = 'Threaded Message Widget';
    }
    """
# Usage example:
# widget = ThreadedMessageWidget()
# display(widget)


In [ ]:
widget = AsyncioMessageWidget()
display(widget)

In [ ]:
#r= await widget.sendGetValue("messaged")

In [ ]:
widget.sendValue("sasa5")
#r=await widget.get_reply_async(5)

In [ ]:
widget.last_response

In [ ]:
r=await widget.get_reply_async(5)

In [ ]:
widget.last_response




In [ ]:
import asyncio
from anywidget import AnyWidget
from traitlets import Unicode, observe

class ThreadedMessageWidget(AnyWidget):
    message = Unicode('').tag(sync=True)
    reply = Unicode('').tag(sync=True)

    def __init__(self):
        super().__init__()
        self._reply_future = None
        self.observe(self._on_reply_change, names=['reply'])

    def sendValue(self, msg):
        # Reset reply to ensure a change is detected
        self.reply = ''
        print("Reply reset, preparing to send new message...")  # Debugging

        # Initialize a new Future for this message
        self._reply_future = asyncio.Future()

        # Now send the message
        print("Sending message:", msg)  # Debugging
        self.message = msg

    async def get_reply_async(self, timeout=None):
        try:
            print("Waiting for reply...")  # Debugging
            reply = await asyncio.wait_for(self._reply_future, timeout)
            print("Received reply:", reply)  # Debugging
            return reply
        except asyncio.TimeoutError:
            print("Timeout waiting for reply")  # Debugging
            raise TimeoutError("Timeout waiting for reply")

    def _on_reply_change(self, change):
        # Ensure that the reply change is properly handled
        if self._reply_future and not self._reply_future.done():
            print("Reply received and future set:", change['new'])  # Debugging
            self._reply_future.set_result(change['new'])  # Complete the Future with the new reply
        else:
            print("No active future or future already done")  # Debugging

# Example usage
#widget = ThreadedMessageWidget()
#display(widget)

async def send_and_receive_async(widget, message):
    widget.sendValue(message)
    try:
        reply = await widget.get_reply_async(timeout=10)  # Await and get the actual reply
        print(f"Received reply: {reply}")
    except TimeoutError:
        print("Timeout waiting for reply")

# Running the async function in the notebook
#await send_and_receive_async(widget, "Hello, Widget!")

In [ ]:
widget = ThreadedMessageWidget()
display(widget)

In [ ]:
r = await send_and_receive_async(widget, "Hello, Widget!")

In [ ]:
from jupyter_anywidget_window_ai import windowai_panel, windowai_inline, windowai_headless

In [ ]:
w = windowai_panel()

In [ ]:
w.update_capabilities()
# We may need to wait a moment
# for the capabilities to be updated

In [ ]:
w.get_capabilities()

In [ ]:
w.create_session(system_prompt="You are a teacher" , temperature=0.8, topK=5)

In [ ]:
w.prompt("Hello, how are you?")

In [ ]:
w.topK=8

In [ ]:
w.destroy()

In [ ]:
from jupyter_anywidget_window_ai import windowai_panel

In [ ]:
wp = windowai_panel(speak_msg=True,)

In [ ]:
wp.create_session(system_prompt="You are a helpful assistant.")


In [ ]:
wp.prompt("Hello, how are you?")

In [ ]:
wp.get_latest_output()

In [ ]:
wp.destroy()

In [ ]:
#%pip install jupyter_ui_poll
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
import time
import uuid

class BlockingWidget(widgets.VBox):
    def __init__(self):
        super().__init__()
        self.send_button = widgets.Button(description="Send Message")
        self.input_text = widgets.Text(placeholder="Enter your message")
        self.output_area = widgets.Output()
        self.replies = {}
        
        self.send_button.on_click(self.on_send_click)
        
        self.children = [self.input_text, self.send_button, self.output_area]
    
    def on_send_click(self, b):
        msg = self.input_text.value
        msg_id = self.send(msg)
        reply = self.get_reply(msg_id)
        with self.output_area:
            print(f"Reply received: {reply}")
    
    def send(self, msg):
        msg_id = str(uuid.uuid4())
        with self.output_area:
            print(f"Sending message: {msg} (ID: {msg_id})")
        return msg_id
    
    def get_reply(self, msg_id):
        start_time = time.time()
        with ui_events() as ui_poll:
            while msg_id not in self.replies:
                ui_poll(10)
                time.sleep(0.1)
                
                # Simulate a reply after 3 seconds
                if time.time() - start_time > 3:
                    self.replies[msg_id] = f"Reply to: {msg_id}"
        
        reply = self.replies.pop(msg_id)
        return reply

# Usage
widget = BlockingWidget()
display(widget)

In [ ]:
msg_id = widget.send("hello?")

In [ ]:
from anywidget import AnyWidget
from traitlets import Unicode, Float
from jupyter_ui_poll import ui_events

import time

class BlockingMessageWidget(AnyWidget):
    message = Unicode('').tag(sync=True)
    reply = Unicode('').tag(sync=True)
    progress = Float(0.0).tag(sync=True)
    
    def __init__(self):
        super().__init__()

    def send_message(self, msg):
        self.reply = ''
        self.progress = 0.0
        self.message = ''
        self.message = msg

    def blocking_reply(self, timeout=None):
        start_time = time.time()
        with ui_events() as ui_poll:
            while not self.reply:
                ui_poll(10)
                if timeout and time.time() - start_time > timeout:
                    raise TimeoutError("Reply not received within the specified timeout.")
                time.sleep(0.1)
            return self.reply


    _esm = """
    export function render({ model, el }) {
        el.innerHTML = `
            <div>
                <p>Message: <span id="message"></span></p>
                <p>Reply: <span id="reply"></span></p>
                <progress id="progress" value="0" max="100"></progress>
            </div>
        `;

        const messageEl = el.querySelector('#message');
        const replyEl = el.querySelector('#reply');
        const progressEl = el.querySelector('#progress');

        const processMessage = (msg) => {
            return new Promise(resolve => {
                let progress = 0;
                const interval = setInterval(() => {
                    progress += 10;
                    model.set('progress', progress / 100);
                    model.save_changes();
                    if (progress >= 100) {
                        clearInterval(interval);
                        resolve(`Processed: ${msg}`);
                    }
                }, 200);
            });
        };

        model.on('change:message', async () => {
            const msg = model.get('message');
            if (msg) {
                messageEl.textContent = msg;
                replyEl.textContent = 'Processing...';
                const reply = await processMessage(msg);
                model.set('reply', reply);
                model.save_changes();
            }
        });

        model.on('change:reply', () => {
            replyEl.textContent = model.get('reply');
        });

        model.on('change:progress', () => {
            progressEl.value = model.get('progress') * 100;
        });
    }
    """

In [ ]:
widget = BlockingMessageWidget()
display(widget)


In [ ]:
# Synchronous usage
widget.send_message("Hellosd, World!")
reply = widget.blocking_reply(timeout=10)
print(f"Received reply: {reply}")

In [ ]:
reply = widget.blocking_reply(timeout=10)
print(f"Received reply: {reply}")

In [ ]:
from jupyter_anywidget_window_ai import windowai_inline
wi = windowai_inline()
wi.create_session(system_prompt="You are a helpful assistant.")


In [ ]:
wi.prompt("Hello, how are you?")

In [ ]:
from jupyter_anywidget_window_ai import windowai_headless
wh = windowai_headless(speak_msg=True)
wh.create_session(system_prompt="You are a helpful assistant.")


In [ ]:
wh.prompt("Hello, how are you?")

In [ ]:
wh.destroy()

In [ ]:
wh.get_latest_output()

In [ ]:
from jupyter_anywidget_window_ai import WindowAIWidget

w= WindowAIWidget()
display(w)

In [ ]:
w.create_session(system_prompt="You are a helpful assistant.")


In [ ]:
w.prompt_streaming("Hello, how are you?")

In [ ]:
response_future = w.prompt("Hello, how are you?")


In [ ]:
w.get_latest_output()

In [ ]:
w.prompt_streaming("What are you?")

In [ ]:
z = w.prompt("What are you?")


In [ ]:
def handle_response(future):
    print("AI response:", future.result())

response_future = w.prompt("Hello, how are you?")
response_future.add_done_callback(handle_response)

In [ ]:
from jupyter_anywidget_window_ai import Widget
W=Widget()
W